<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet6%20(ANN)%20PBL/Tugas%20Praktikum%20(JS6)%20PBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 9.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 29.2 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520033 sha256=83abd8756b39490a39c652f354a557b90e6a77fd67d9a4cc766b9d0f2e0636b3
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=f6035cd663e6ba3a73f5c6e0facec3a8bbe4a042b8549727e89ce627cd7bd45b
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

print("Memuat dataset cafes & reviews dari folder Android...")

# Path file sesuai posisi foldermu di HP
cafes_path = '/storage/emulated/0/Android/cafes.csv'
reviews_path = '/storage/emulated/0/Android/reviews.csv'

cafes = pd.read_csv(cafes_path)
reviews = pd.read_csv(reviews_path)

# Agregasi data reviews per cafe
reviews_agg = reviews.groupby('cafe_id').agg(
    avg_review_rating=('rating', 'mean'),
    total_likes=('likes', 'sum')
).reset_index()

merged = pd.merge(cafes, reviews_agg, on='cafe_id', how='left').fillna(0)

# Ambil fitur numerik & tentukan sampel 1000 agar optimal di HP
feature_cols = ['latitude', 'longitude', 'rating', 'review_count', 'avg_review_rating']
raw_data = merged[feature_cols].values.astype(np.float32)

np.random.seed(42)
n_samples = 1000
sample_indices = np.random.choice(len(raw_data), n_samples, replace=True)
X = raw_data[sample_indices]
d = X.shape[1]

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
k = 10

print(f"Dataset berhasil dimuat! Jumlah sampel: {n_samples}, Dimensi: {d}D\n")

# Inisialisasi variabel waktu
time_exact = 0.0
time_annoy = 0.0
time_hnsw = 0.0
time_faiss = 0.0

# 1. Exact NN (brute-force)
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
_, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# 2. Annoy
try:
    from annoy import AnnoyIndex
    start = time.time()
    index_annoy = AnnoyIndex(d, 'euclidean')
    for i, v in enumerate(X_scaled):
        index_annoy.add_item(i, v)
    index_annoy.build(10)
    idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
    time_annoy = time.time() - start
    print(f"Annoy done in {time_annoy:.3f} s")
except Exception as e:
    print(f"Annoy : Skipped (Module not found/Unsupported)")

# 3. HNSW
try:
    import hnswlib
    start = time.time()
    p_hnsw = hnswlib.Index(space='l2', dim=d)
    p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
    p_hnsw.add_items(X_scaled)
    p_hnsw.set_ef(200)
    idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
    time_hnsw = time.time() - start
    print(f"HNSW done in {time_hnsw:.3f} s")
except Exception as e:
    print(f"HNSW : Skipped (C++ Build Error di Android)")

# 4. FAISS IVF
try:
    import faiss
    start = time.time()
    quantizer = faiss.IndexFlatL2(d)
    nlist = 20
    index_faiss = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
    index_faiss.train(X_scaled)
    index_faiss.add(X_scaled)
    index_faiss.nprobe = 5
    _, idx_faiss = index_faiss.search(X_scaled, k)
    time_faiss = time.time() - start
    print(f"FAISS IVF done in {time_faiss:.3f} s")
except Exception as e:
    print(f"FAISS : Skipped (C++ Build Error di Android)")

# ===============================
# Ringkasan Waktu (Format Output Konsisten)
# ===============================
print("\n=== Ringkasan Waktu PyDroid 3 HP (detik) ===")
print(f"Exact NN : {time_exact:.3f}" if time_exact > 0 else "Exact NN : Unsupported")
print(f"Annoy    : {time_annoy:.3f}" if time_annoy > 0 else "Annoy    : Unsupported")
print(f"HNSW     : {time_hnsw:.3f}" if time_hnsw > 0 else "HNSW     : Unsupported")
print(f"FAISS    : {time_faiss:.3f}" if time_faiss > 0 else "FAISS    : Unsupported")

Dataset siap! Jumlah sampel: 10000, Dimensi: 5D

Exact NN done in 1.244 s
Annoy done in 0.864 s
HNSW done in 2.456 s
FAISS IVF done in 0.219 s

=== Ringkasan Waktu (detik) ===
Exact NN : 1.244
Annoy    : 0.864
HNSW     : 2.456
FAISS    : 0.219
